# Hello!

This worksheet is designed as learning material for an undergraduate student with a strong understanding of calculus and some knowledge of orbital mechanics, but no prior experience with computational physics. It covers the basics of taking definite integrals and shows you how to start coding your own orbital dynamics simulations.

In [11]:
# This cell handles the importation of the software libraries we'll need to run
# our simulations and visualize the resulting data.

!pip install -U plotly
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider


*Let's start with the basics.*

The essential equation in orbital dynamics is Newton's Law of Gravitation.

$F = \frac{m_1 \times m_2}{r^2}G$

And of course, we can construct the equation for the acceleration experienced by a body under the influence of this force my dividing by its mass. I'll add in directionality here with some vectors.

$\vec{a}_{m_1} = \frac{\vec{F}}{m_1} = -\frac{m_2}{r^2}G(\hat{r})$

Here, $\hat{r}$ points from $m_1$ towards $m_2$, but it would be useful to instead express $\vec{r}$ from an inertial reference frame. Say we define the centre of mass of the system as $\vec{R} = \frac{m_1 r_1 + m_2 r_2}{m_1 + m_2}$ It can be shown that $\frac{d^2\vec{R}}{dt^2} = 0$, so the barycenter acts as an inertial reference frame, and positions defined from this point of reference will allow us to take advantage of some physical constants. We can rearrange this into the general equation of motion for $m_1$.

$\frac{d^2 \vec{r}}{dt^2} + \frac{G(m_1+m_2)}{r^3}\vec{r} = 0$

The extra $r$ in the denominator of the second term comes from us expressing the direction as the non-unit vector $\vec{r}$ rather than $\hat{r}$. You might recognize this as an analytically solvable ODE (hint: it's separable, and angular momentum $L = \mu{}\vec{r}\times{}\frac{d\vec{r}}{dt}$ where $\mu{} = \frac{m_1 m_2}{m_1 + m_2}$ is constant). Depending on the energy of the system, the solution to this problem will trace a conic section.

Generally, the equation of motion is of the form $r(\theta{}) = \frac{\frac{L^2}{\mu{} k}}{1+ecos(\theta{}-\theta{}_0)}$, where $e$ $\theta{}_0$, and L provide the degrees of freedom required to match the solution to any initial conditions. The cell below provides a visual representation of what the solution looks like depending on the parameters of the problem, feel free to play around with the parameters to see how they effect the paths of the bodies. 

In [ ]:
G = 1.0

def orbit(e=0.3, theta0=0.0, L=1.0, m1=1.0, m2=1.0):
    mu = (m1 * m2) / (m1 + m2)
    k = G * (m1 + m2)
    p = L**2 / (mu * k)

    if e > 1:
        theta_max = np.arccos(-1 / e)
        theta = np.linspace(theta0 - theta_max, theta0 + theta_max, 1000)
    else:
        theta = np.linspace(0, 2*np.pi, 1000)
    
    r = p / (1 + e * np.cos(theta - theta0))
    r[r < 0] = np.nan

    x = r * np.cos(theta)
    y = r * np.sin(theta)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='Orbit'))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers', name='Focus'))

    fig.update_layout(
        width=600,
        height=600,
        title=f"Orbit (e={e:.2f})"
    )

    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()

interact(
    orbit,
    e=FloatSlider(min=0.0, max=2.0, step=0.01, value=0.3),
    theta0=FloatSlider(min=0.0, max=2*np.pi, step=0.01, value=0.0),
    L=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0),
    m1=FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0),
    m2=FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0),
)

interactive(children=(FloatSlider(value=0.3, description='e', max=2.0, step=0.01), FloatSlider(value=0.0, desc…

<function __main__.orbit(e=0.3, theta0=0.0, L=1.0, m1=1.0, m2=1.0)>

You might notice that only one of the bodies has a traced path in this plot. *THis is because we can solve for the paths of the bodies (but not the positions w.r.t. time) independently.* We have reduced the two body problem to two *reduced* two body problems; $m_1$ and the barycentre, and $m_2$ and the barycentre. These problems are called reduced because one of the bodies, the barycentre, is considered to be immobile.